# 02 — Signal Processing: Bandpass Filtering

Remove baseline wander and high-frequency noise from the raw ECG while preserving QRS morphology, using a Butterworth bandpass filter. No R-peak detection yet.

## Load the record

Same record as `01_data_exploration.ipynb` (each notebook loads its own data so it can run independently).

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

RECORD_NAME = "100"

record = wfdb.rdrecord(RECORD_NAME, pn_dir="mitdb")
fs = record.fs
lead_index = record.sig_name.index("MLII") if "MLII" in record.sig_name else 0
ecg_signal = record.p_signal[:, lead_index]
time = np.arange(len(ecg_signal)) / fs

## Bandpass filter

- **Band (0.5-40 Hz)**: baseline wander lives below ~0.5 Hz, real QRS energy is
  concentrated below ~25 Hz, and muscle/electrical noise and mains
  interference sit above 40 Hz — this band keeps the ECG content and drops
  both noise sources on either side.
- **Order 4**: controls how sharply the filter transitions from pass to
  block at the cutoffs. A standard, well-tested choice for ECG — sharp
  enough to be effective, gentle enough not to distort the waveform.
- **`filtfilt` instead of a single-pass filter**: a normal filter shifts the
  signal in time (phase delay), which would move the QRS to a different
  sample index. `filtfilt` runs the filter forward then backward, cancelling
  that shift out — important since we'll later compare detected peak
  *indices* against annotation indices with only a small tolerance.

In [ ]:
def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    """Zero-phase Butterworth bandpass filter for ECG."""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)


filtered_signal = bandpass_filter(ecg_signal, lowcut=0.5, highcut=40, fs=fs, order=4)

## Raw vs filtered, same time window

Reusing the same 3-second window from `01_data_exploration.ipynb` so this is
a direct before/after comparison.

**What to check:** the QRS spikes should still be there, clearly, at
essentially the same sharpness (or sharper). The baseline in the filtered
version should sit flat around 0 mV instead of drifting. Fine jitter between
beats should be visibly reduced. If the QRS peaks looked shrunken, delayed,
or distorted, that would mean the filter is too aggressive or the cutoffs
are wrong — not the case here, but this is exactly the kind of check that
catches it.

In [ ]:
zoom_start, zoom_end = 60, 63  # seconds
mask = (time >= zoom_start) & (time < zoom_end)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(time[mask], ecg_signal[mask], color="black", linewidth=1.0)
axes[0].set_title("Raw ECG")
axes[0].set_ylabel("Amplitude (mV)")

axes[1].plot(time[mask], filtered_signal[mask], color="tab:blue", linewidth=1.0)
axes[1].set_title("Filtered ECG (0.5-40 Hz Butterworth bandpass, order 4)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude (mV)")

fig.tight_layout()
fig.savefig("../results/figures/03_raw_vs_filtered.png", dpi=120)
plt.show()